In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
!pip install catboost

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostClassifier

csv_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'], bins=30, edgecolor='black')
plt.title('Target Distribution')
plt.xlabel('Target')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns="Order_ID", axis=1)
df.head()

In [ ]:
# Task 2: Write your code here:
# Checking the missing values
# print("Missing values per column:")
# print(df.isnull().sum())
# print(f"\nTotal missing: {df.isnull().sum().sum()}")


# We need to copy the data before cleaning
df_clean = df.copy()

# I will drop the missing values in Weather, Traffic Level, Time_of_Day, and Courier_experience_yrs
cols_to_drop = ['Weather', 'Traffic_Level', 'Time_of_Day','Courier_Experience_yrs']
df_clean = df_clean.dropna(subset=cols_to_drop)


# I will fill the missing target values with the mean:
df_clean['Delivery_Time'] = df_clean['Delivery_Time'].fillna(df_clean['Delivery_Time'].mean())


In [ ]:
# Task 3: Write your code here:

print(f"Duplicates: {df_clean.duplicated().sum()}")

# We have 489 duplicates
df_clean.drop_duplicates(inplace=True)

In [ ]:
# Task 4: Write your code here:
categorical_cols = df_clean.select_dtypes(include=['object']).columns
categorical_cols = categorical_cols.drop('Traffic_Level')

# Traffic Level is ordinal and so label encoder is needed
le = LabelEncoder()
df_clean['Traffic_Level'] = le.fit_transform(df_clean['Traffic_Level'])

# One hot encoding for nominal columns
df_clean = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True, dtype=int)

In [ ]:
# Task 5: Write your code here:
X = df_clean.drop('Delivery_Time', axis=1).astype(float)
y = df_clean['Delivery_Time']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Task 6: Write your code here:
# Prediction task doesn't have classes or imbalance

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop('Delivery_Time', axis=1).astype(float)
y = df_clean['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:

scores = []

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_train, X_test = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[val_idx]
    print(f"Fold {fold+1}: Train={len(train_idx)}, Val={len(val_idx)}")

    model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    scores.append(mae)


score = sum(scores)/len(scores)

print(f"The average score of all folds is: {score}")


In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.hist(y_pred)

In [ ]:
# Task Bonus: Write your code here:

# Defining the models
sklearn_models = {
  "RandomForest": RandomForestRegressor(
      n_estimators=100,
      max_depth=10,
      random_state=42)
  ,
  "CatBoost": CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
}

In [ ]:
n_splits = 5

scores = []

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  random_f = sklearn_models["RandomForest"]
  catboost = sklearn_models["CatBoost"]

  random_f.fit(X_train, y_train)
  catboost.fit(X_train, y_train)

  y_pred_random = model.predict(X_test)
  y_pred_catboost = model.predict(X_test)

  y_pred_avg = (y_pred_random + y_pred_catboost) / 2

  mae = mean_absolute_error(y_test, y_pred_avg)

  scores.append(mae)


In [ ]:
print(sum(scores) / len(scores))